# When does fine-tuning beat the API?
Fine-tune Qwen3-8B with QLoRA on clinical-note extraction, then compare it with the untuned model, Claude Haiku 5.5 and Claude Sonnet 5.5 on accuracy, cost and speed.

**Runtime:** a 24 GB+ GPU (L4, A10G or A100). A T4 is too small to serve an 8B model with vLLM.
**Time:** about 1.5–2.5 hours end to end. **Cost:** GPU time plus about $2–3 of Claude API calls.

In [ ]:
# Stop early if the GPU is too small (a T4 has ~15 GB: not enough to train AND serve an 8B model)
import subprocess
name, mem = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True).strip().split(", ")
print(f"GPU: {name}, {int(mem)/1024:.1f} GB")
assert int(mem) >= 20000, "GPU too small. Runtime > Change runtime type > pick A100 or L4, then Runtime > Run all again."

## 1. Get the code and install

In [ ]:
# Replace with your own repo after you push it, or upload the zip and unzip it here.
%cd /content
![ -d finetune-vs-api ] || git clone https://github.com/gd0399/finetune-vs-api.git
%cd /content/finetune-vs-api
!git pull -q
!pip install -q -r requirements-gpu.txt
!pip uninstall -y -q torchao   # Colab's old torchao breaks peft's adapter merge

## 2. Build the dataset
2,000 training notes in 3 styles; two test sets: familiar styles (ID) and new styles with unseen conditions and drugs (OOD).

In [ ]:
!python -m ftva.cli generate
!head -c 700 data/test_ood.jsonl

## 3. Fine-tune with QLoRA (about 45–90 min)

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python train/train_lora.py --model Qwen/Qwen3-8B --out adapters/qwen3-8b-lora

In [ ]:
!python train/merge_lora.py --base Qwen/Qwen3-8B --adapter adapters/qwen3-8b-lora --out merged/qwen3-8b-ft

## 4. Evaluate the base and fine-tuned models on vLLM
`--workers 64` sends 64 requests at once, so the measured notes/hour reflects a busy production server.

In [ ]:
from ftva import serve
import subprocess

for model, name in [("Qwen/Qwen3-8B", "qwen3-8b-base"), ("merged/qwen3-8b-ft", "qwen3-8b-ft")]:
    proc = serve.start(model, name)
    try:
        subprocess.run(["python", "-m", "ftva.cli", "eval", "--spec", f"openai:{name}@http://localhost:8000/v1",
                        "--label", name, "--workers", "64"], check=True)
    finally:
        serve.stop(proc)

## 5. Evaluate Claude Haiku 5.5 and Sonnet 5.5

In [ ]:
import os, getpass
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
!python -m ftva.cli eval --spec anthropic:claude-haiku-5-5 --label haiku-5.5 --workers 8
!python -m ftva.cli eval --spec anthropic:claude-sonnet-5-5 --label sonnet-5.5 --workers 8

## 6. Report
Set `GPU_HOURLY` to what you actually pay per hour for **this** GPU: the break-even depends on it.

In [ ]:
GPU_HOURLY = 3.52   # A100 list price (Google Colab Enterprise); <-- your provider's price for the GPU this notebook ran on
!python -m ftva.cli report --gpu-hourly {GPU_HOURLY}
from IPython.display import Image, display
display(Image("results/accuracy.png")); display(Image("results/break_even.png"))

Also try `--batch` (Batch API pricing, 50% off) to see how the break-even moves if results can wait up to 24 hours.

In [ ]:
!python -m ftva.cli report --gpu-hourly {GPU_HOURLY} --batch | tail -8

## 7. Save results
Download `results/` (and optionally `adapters/`) before the runtime shuts down.

In [ ]:
!zip -qr results.zip results adapters && ls -lh results.zip